In [2]:
# Import all required modules

import torch
import numpy as np
import sounddevice as sd
import scipy.io.wavfile as wav
import datetime
import librosa

from transformers import (
    AutoProcessor,
    AutoModelForSpeechSeq2Seq,
    pipeline
)

In [3]:
# Load Whisper ASR model

print("Loading ASR model...")

asr_model_id = "openai/whisper-tiny"

asr_processor = AutoProcessor.from_pretrained(asr_model_id)
asr_model = AutoModelForSpeechSeq2Seq.from_pretrained(asr_model_id)

device = "cuda" if torch.cuda.is_available() else "cpu"
asr_model.to(device)

print("ASR model loaded successfully!")

Loading ASR model...
ASR model loaded successfully!


In [4]:
# Load Text-to-Speech model

print("Loading TTS model...")

tts = pipeline(
    "text-to-speech",
    model="facebook/mms-tts-eng"
)

print("TTS ready!")

Loading TTS model...
TTS ready!


In [11]:
# Record audio from microphone

def record_audio(duration=5, sample_rate=16000):
    print("🎤 Speak now...")

    audio = sd.rec(
        int(duration * sample_rate),
        samplerate=sample_rate,
        channels=1,
        dtype='float32'   
    )

    sd.wait()
    return audio.flatten(), sample_rate

In [12]:
# Convert speech to text

def speech_to_text(audio, sr):
    inputs = asr_processor(
        audio,
        sampling_rate=sr,
        return_tensors="pt",
        padding=True  
    )

    input_features = inputs.input_features.to(device)

    with torch.no_grad():
        predicted_ids = asr_model.generate(
            input_features,
            language="en"  
        )

    text = asr_processor.batch_decode(
        predicted_ids,
        skip_special_tokens=True
    )[0]

    return text.strip()

In [ ]:
# Generate response

def generate_response(text):

    text = text.lower()

    # Handle unclear input
    if len(text.strip()) < 2:
        return "I couldn't clearly understand. Please try again."

    # Study assistant logic
    if "cloud computing" in text:
        return "Cloud computing means storing and accessing data over the internet."

    elif "ai" in text:
        return "Artificial Intelligence allows machines to think and learn like humans."

    elif "python" in text:
        return "Python is a programming language used for AI, web development, and data science."

    elif "time" in text:
        return "Current time is " + datetime.datetime.now().strftime("%H:%M")

    elif "date" in text:
        return "Today is " + str(datetime.date.today())

    elif "hello" in text:
        return "Hello! I am your Voice Study Assistant."

    else:
        return "I heard: " + text + ". Try asking a study-related question."

In [13]:
# Convert text to speech

def text_to_speech(text, filename="output.wav"):
    print(" Assistant:", text)

    tts_output = tts(text)

    audio = np.array(tts_output["audio"])   
    sr = tts_output["sampling_rate"]

    # ensure mono format
    audio = np.squeeze(audio).astype(np.float32)

    # safe playback
    try:
        sd.play(audio, sr)
        sd.wait()
    except Exception as e:
        print("Audio playback error:", e)

    # save file
    wav.write(filename, sr, audio)

    print(" Saved as:", filename)

In [14]:
# Load audio file 
def load_audio_file(path):
    audio, sr = librosa.load(path, sr=16000)
    return audio, sr

In [15]:
# Full Voice AI pipeline

print(" Voice AI Study Assistant Started")

# Step 1: Record audio
audio, sr = record_audio(duration=5)

# OR use file:
# audio, sr = load_audio_file("sample.wav")

# Step 2: Speech to text
text = speech_to_text(audio, sr)
print(" You said:", text)

# Step 3: Generate response
response = generate_response(text)

# Step 4: Text to speech
text_to_speech(response)

print(" Process completed!")

 Voice AI Study Assistant Started
🎤 Speak now...


You have passed language=en, but also have set `forced_decoder_ids` to [[1, None], [2, 50359]] which creates a conflict. `forced_decoder_ids` will be ignored in favor of language=en.


 You said: AI.
 Assistant: AI stands for Artificial Intelligence, enabling machines to think and learn.
 Saved as: output.wav
 Process completed!
